## Computing environment

Lean Stable 4.33.1 

The goal is to prove the

* Conjecture. a(n) = A060014(n) = (sum of orders of all permutations of n letters) is, for every m > 0, eventually periodic mod m. [Isaac Saffold, Dec 01 2019]

* Check the correctness of the proof proposed by N.L.S.

This file was written by Claude Sonnet 5 and prompted by P.L.

# SumOfOrders

## First Check

The overall strategy (of the proof proposed by NLS) is sound and the conjecture is almost certainly true, but the argument as written has one real error — in the identification of the "alphabet" of reachable `g`-values — which invalidates the specific bound it states (though not the qualitative conclusion).

**What's correct**

- **Setup.** `b(n,g) = Σ_{p∈S_n} lcm(ord(p), g)` does satisfy Heinz's recurrence. Decomposing by the cycle containing the last element, the number of ways to choose+cyclically-order the rest of that cycle is `C(n-1,j-1)·(j-1)! = ff(n-1,j-1)`, and the remaining `n-j` elements recurse with `g` replaced by `lcm(g,j)`. So `a(n)=b(n,1)=A060014(n)`. ✓.

- **The "build" combinatorics.** The alternate derivation via `Σ_i C(n-j,i)C(j-1,i) = C(n-1,j-1)` is a genuine Vandermonde identity (using `C(j-1,i)=C(j-1,j-1-i)`), so it does land on the right coefficient — though it's a much more roundabout route than the standard cycle-decomposition argument, and the write-up doesn't really justify why that particular sum counts the described "swap" construction. Not fatal, since the coefficient is independently verifiable.

- **The pruning argument.** Since `(j-1)! | ff(n-1,j-1)`, any single step using cycle-length `j>m` makes that step's factor — and hence the whole nested product for that path — divisible by `m`. So mod `m`, only decomposition paths using parts `j≤m` at *every* level survive. I verified this numerically: restricting the recursion to `j≤3` reproduces `a(10) mod 3` exactly.

**The error**

The argument claims the surviving values of `g` are exactly row `m` of A256553 — i.e. the `A009490(m)` distinct *orders of permutations of exactly m elements* (lcm's of partitions summing to `m`). That's wrong. Since the recursion accumulates `g ← lcm(g,j)` over arbitrarily many steps (as `n→∞`), with no constraint that the `j`'s sum to `m`, the reachable set is actually

$$U_m = \{\operatorname{lcm}(T): T\subseteq\{1,\dots,m\}\},$$

which equals **all divisors of `lcm(1,...,m) = A003418(m)`** (every divisor `d` of `A003418(m)` has each prime-power component ≤ `m` automatically, so `d = lcm` of its own prime-power parts, a subset of `{1,...,m}`; conversely every such lcm divides `A003418(m)`). This is generally strictly larger than row `m` of A256553, whose entries additionally need the parts to *sum* to exactly `m`.

Concrete check (m=3): row 3 of A256553 is `{1,2,3}` (`A009490(3)=3`). But computing `a(10) mod 3` via the pruned recursion, the value `g=6=lcm(2,3)` genuinely occurs (I confirmed this by instrumenting the recursion — `b(5,6)` is visited). `6` doesn't appear in A256553 until row 5, since a single permutation needs 5 elements to realize order 6. Divisors of `lcm(1,2,3)=6` are `{1,2,3,6}` — size 4, matching what's actually needed.

**Consequence**

The stated bound `m^(1+m·A009490(m))` is too small / uses the wrong quantity — it should be `m^(1+m·d(A003418(m)))` (`d` = divisor-counting function), since that's the correct finite alphabet size for `g`. This doesn't break the *argument's logic*, though — `d(A003418(m))` is still finite for each `m`, so the pigeonhole/finite-state conclusion (eventual periodicity of `a(n) mod m`) still goes through with the corrected bound.

**Summary:** the recurrence, the polynomial-periodicity-in-`n mod m` point, and the "only `j≤m` survives" pruning are all correct. The identification of the reachable `g`-alphabet with A256553/A009490 is incorrect — it should be "divisors of `A003418(m)`" — which invalidates the specific stated state-count/period bound, but a corrected version of the same argument (swap in the right alphabet size) still proves the conjecture.

## Cleaned up

**Conjecture.** A060014 (sum of orders of all permutations of $n$ letters) eventually becomes cyclic mod $m$, for every $m$.

**Setup.** For $n\ge0$ and $g\ge1$, let
$$b(n,g)=\sum_{p\in S_n}\operatorname{lcm}(\operatorname{ord}(p),g),$$
with $S_0$ containing only the empty permutation of order 1, so $b(0,g)=g$. Then $a(n):=b(n,1)=\sum_{p\in S_n}\operatorname{ord}(p) = A060014(n)$.

**Recurrence.** Fix $n\ge1$ and split $S_n$ by $j$, the length of the cycle containing the element $n$. There are $\binom{n-1}{j-1}$ ways to choose the other $j-1$ cycle members and $(j-1)!$ ways to arrange all $j$ elements cyclically; the remaining $n-j$ elements form an independent permutation $q\in S_{n-j}$, and $\operatorname{ord}(p)=\operatorname{lcm}(j,\operatorname{ord}(q))$. Summing,
$$b(n,g)=\sum_{j=1}^n \operatorname{ff}(n-1,j-1)\,b(n-j,\operatorname{lcm}(g,j)),\qquad \operatorname{ff}(n,k):=\tfrac{n!}{(n-k)!}=\binom{n}{k}k!. \tag{$\star$}$$
This is exactly Heinz's program.

**Lemma 1 (coefficients are periodic in $n$ mod $m$).** For fixed $j$, $\operatorname{ff}(n-1,j-1)=(n-1)(n-2)\cdots(n-j+1)$ is an integer-coefficient polynomial in $n$. Any such polynomial satisfies $P(n+m)\equiv P(n)\pmod m$ (binomial expansion of $(n+m)^k-n^k$ has every term but $n^k$ divisible by $m$). So $\operatorname{ff}(n-1,j-1)\bmod m$ depends only on $n\bmod m$.

**Lemma 2 (pruning).** If $j>m$, then $m\mid\operatorname{ff}(n-1,j-1)$, because $(j-1)!\mid\operatorname{ff}(n-1,j-1)$ and $m\mid(j-1)!$ once $j-1\ge m$. Consequently, if we fully unroll $(\star)$ into a sum over compositions $n=j_1+\cdots+j_r$, any composition using some $j_i>m$ contributes a multiple of $m$ to $b(n,1)$ (that factor alone kills the whole product of coefficients along that path). So **at every level of the recursion**, mod $m$ we may truncate the sum to $j\le m$:
$$b(n,g)\equiv\sum_{j=1}^{\min(n,m)}\operatorname{ff}(n-1,j-1)\,b(n-j,\operatorname{lcm}(g,j))\pmod m,\qquad n\ge1. \tag{$\star_m$}$$

**Lemma 3 (the correct finite alphabet for $g$).** Let $L=L_m:=\operatorname{lcm}(1,\dots,m) = $ A003418$(m)$. If $g\mid L$ and $1\le j\le m$, then $j\mid L$, so $\operatorname{lcm}(g,j)\mid L$. Since we start from $g=1$ and, by $(\star_m)$, only ever take $\operatorname{lcm}$ with values $\le m$, **every value of $g$ that can occur is a divisor of $L$.** Conversely every divisor $d\mid L$ does occur for large enough $n$: writing $d=\prod p^{a_p}$, each $p^{a_p}$ divides $L$ and hence is $\le m$ (by construction $L$'s $p$-part is the largest power of $p$ not exceeding $m$), so $d=\operatorname{lcm}$ of the set $\{p^{a_p}\}\subseteq\{1,\dots,m\}$, reachable as a path using each of these once. So the exact set of relevant $g$'s is $\{d : d\mid L_m\}$, of size
$$D(m):=d(L_m)=d(\text{A003418}(m)),$$
the ordinary divisor-counting function ($A000005$) applied to $A003418(m)$, i.e. $\tau(L)=A056793$, **not** the row-length $A009490(m)$ of A256553 (that counts orders of permutations of *exactly* $m$ points, i.e. lcm's of partitions *summing to* $m$, which is a strictly smaller set, e.g. for $m=3$ it misses $g=6=\operatorname{lcm}(2,3)$, which genuinely occurs).

**Lemma 4 (finite-state machine).** Let $g_1=1,\dots,g_D$ ($D=D(m)$) enumerate the divisors of $L_m$. For $n\ge m-1$ define the state
$$S(n)=\Big(n\bmod m,\ \big(b(n-i,g_k)\bmod m\big)_{0\le i<m,\ 1\le k\le D}\Big).$$
By Lemmas 1–3, $(\star_m)$ expresses each $b(n+1,g_k)\bmod m$ as a function of $n\bmod m$ and the values $b(n+1-j,\cdot)\bmod m$ for $j=1,\dots,m$ — all of which are already recorded in $S(n)$. So $S(n+1)$ is a deterministic function of $S(n)$. The state space has size at most
$$m\cdot m^{mD(m)} = m^{\,1+m\,D(m)}.$$
By pigeonhole, among $S(m-1),S(m),\dots,S(m-1+m^{1+mD(m)})$ two must coincide, say $S(n_1)=S(n_2)$, $n_1<n_2$; determinism then forces $S(n+ (n_2-n_1))=S(n)$ for all $n\ge n_1$.

**Conclusion.** In particular $a(n)\bmod m = b(n,1)\bmod m$ (the $g_1=1$ component of the state) is eventually periodic, with pre-period and period both at most $m^{1+m\,D(m)}$, $D(m)=d(A003418(m))$. As this holds for every $m$, A060014 eventually becomes cyclic mod $m$ for all $m$. $\blacksquare$


----------------------

## LEAN / STATUS / HONESTY NOTE

A full, gap-free formalization of this would be a serious undertaking — it touches group theory of $S_n$, a bijective proof of the recurrence, general dynamics, and modular arithmetic on falling factorials. I don't have a Lean/Mathlib toolchain available here to compile-check anything, so what I'm giving you is a careful **draft skeleton**: definitions and theorem statements that faithfully mirror the write-up, with real attempted proofs where the content is genuinely tractable, and `sorry` (clearly commented) where the content is the substantial mathematical work from Lemmas 1–3 and the recurrence-correctness theorem.
/-
  A060014_periodicity.lean

  A Lean 4 / Mathlib skeleton for:

    Conjecture. a(n) = A060014(n) = (sum of orders of all permutations of n letters) is, for every m > 0, eventually periodic mod m.

  This file was written without access to a Lean + Mathlib toolchain in this
  session, so it has NOT been compiled or type-checked. Treat it as a
  structural translation of the informal proof, not a verified artifact.
  Mathlib's exact lemma names drift over time, so names below may need small
  fixes even where the mathematics is right.

  Two pieces are given genuine, "should really work" proof attempts:
    * `pruning`                       (Lemma 2 of the write-up)
    * `eventually_periodic_of_finite'` (the abstract pigeonhole core of Lemma 4)

  Everything that encodes real combinatorial/number-theoretic content specific
  to this problem (the recurrence's correctness, Lemma 1, Lemma 3, and the
  final assembly) is stated precisely and left as `sorry`, with a comment
  pointing back to the corresponding step of the informal argument.
-/


In [ ]:
import Mathlib

open Finset

/- ================================================================
   1. The recurrence b(n,g), and a(n) = b(n,1).
   ================================================================ -/

/-- `b n g = Σ_{p ∈ S_n} lcm(ord p, g)`, defined via the cycle-removal
recurrence (Heinz's program / equation (⋆) of the write-up).
`Nat.descFactorial n i = n * (n-1) * ⋯ * (n-i+1)` is Mathlib's falling
factorial, matching `ff(n,i)` in the write-up. -/
def b : ℕ → ℕ → ℕ
  | 0, g => g
  | n + 1, g =>
      ∑ i ∈ Finset.range (n + 1),
        Nat.descFactorial n i * b (n - i) (Nat.lcm g (i + 1))
termination_by n _ => n
decreasing_by all_goals omega

/-- `a n = A060014(n)`. -/
def a (n : ℕ) : ℕ := b n 1

/-- The combinatorial object `a` is supposed to compute: the sum, over all
permutations of `Fin n`, of their order in the permutation group. -/
noncomputable def sumOrders (n : ℕ) : ℕ :=
  ∑ p : Equiv.Perm (Fin n), orderOf p

/-- **Recurrence correctness** ("Setup" + "Recurrence" sections of the
write-up). Proving this formally means constructing, for each `p : Equiv.Perm
(Fin (n+1))`, the bijective decomposition "the cycle containing the point
`n`, of some length `j`, together with a permutation of the other `n+1-j`
points", and showing the count of such decompositions with cycle length `j`
is `Nat.descFactorial n (j-1)`. This is the one place where the real
group-theoretic combinatorics of the problem lives; it is substantial and is
left here as `sorry`. -/
theorem a_eq_sumOrders (n : ℕ) : a n = sumOrders n := by
  sorry

/- ================================================================
   2. Lemma 1: coefficients are periodic in n mod m.
   ================================================================ -/

/-- **Lemma 1.** For `i ≤ n` (so that no `ℕ`-subtraction truncates),
`Nat.descFactorial n i`, as a function of `n`, only depends on `n mod m`.
Proof idea: cast to `ZMod m`; `Nat.descFactorial n i` is (in this range) the
same as evaluating the integer polynomial `X(X-1)⋯(X-i+1)` at `n`, and
evaluation into `ZMod m` only sees `(n : ZMod m)`, which is unchanged by
adding `m`. -/
theorem descFactorial_periodic_mod (m i n : ℕ) (hi : i ≤ n) :
    Nat.descFactorial (n + m) i ≡ Nat.descFactorial n i [MOD m] := by
  sorry

/- ================================================================
   3. Lemma 2: pruning (j > m contributes 0 mod m). Real attempt.
   ================================================================ -/

/-- **Lemma 2.** If `m < j` then `m ∣ Nat.descFactorial n (j - 1)`, i.e. the
coefficient of the `j`-cycle-removal step is divisible by `m` once `j > m`.
This uses two standard Mathlib facts: `m ∣ (j-1)!` once `m ≤ j-1`, and
`k! ∣ Nat.descFactorial n k` always. -/
theorem pruning (m n j : ℕ) (hm : 0 < m) (hj : m < j) :
    m ∣ Nat.descFactorial n (j - 1) := by
  have h1 : m ≤ j - 1 := by omega
  have h2 : m ∣ (j - 1)! := Nat.dvd_factorial hm h1
  have h3 : (j - 1)! ∣ Nat.descFactorial n (j - 1) :=
    Nat.factorial_dvd_descFactorial n (j - 1)
  exact h2.trans h3

/-- Consequence used in the write-up: mod `m`, `(⋆)` may be truncated to
`j ≤ m` at every level of the recursion (this is the "through any path of
nested sums" step — divisibility of one factor in a product propagates to
divisibility of the whole product). -/
theorem pruning_propagates (m a b c : ℕ) (h : m ∣ b) : m ∣ a * b * c := by
  exact Dvd.dvd.mul_right (Dvd.dvd.mul_left h a) c

/- ================================================================
   4. Lemma 3: the correct alphabet is divisors of lcm(1,...,m).
   ================================================================ -/

/-- `L m = lcm(1, 2, ..., m)`, i.e. `A003418(m)`. -/
def L (m : ℕ) : ℕ := (Finset.Icc 1 m).lcm id

/-- The finite alphabet of reachable `g`-values mod `m`: **all divisors of
`L m`**, of size `d(L m)` — *not* `A009490 m` / row `m` of A256553 (that
undercounts, as the m = 3 example with g = 6 = lcm(2,3) shows). -/
def Alphabet (m : ℕ) : Finset ℕ := (L m).divisors

/-- **Lemma 3.** Every `g` reachable by starting at `g = 1` and repeatedly
taking `lcm` with values `≤ m` is a divisor of `L m`, and every divisor of
`L m` is reachable this way. -/
theorem alphabet_correct (m : ℕ) (hm : 0 < m) :
    ∀ d, d ∈ Alphabet m ↔
      ∃ (T : Finset ℕ), (∀ t ∈ T, 1 ≤ t ∧ t ≤ m) ∧ T.lcm id = d := by
  sorry

/- ================================================================
   5. Lemma 4 (abstract core): finite self-maps are eventually periodic.
      Real attempt — this part is elementary and should be solid modulo
      exact Mathlib lemma names.
   ================================================================ -/

/-- Pigeonhole: the forward orbit of any point under a self-map of a finite
type must repeat. -/
theorem eventually_periodic_of_finite {α : Type*} [Finite α] (f : α → α)
    (x : α) : ∃ N p : ℕ, 0 < p ∧ f^[N + p] x = f^[N] x := by
  obtain ⟨n1, n2, hne, heq⟩ :=
    Finite.exists_ne_map_eq_of_infinite (fun n : ℕ => f^[n] x)
  rcases lt_or_gt_of_ne hne with h | h
  · exact ⟨n1, n2 - n1, by omega, by
      have hEq : n1 + (n2 - n1) = n2 := by omega
      rw [hEq]; exact heq.symm⟩
  · exact ⟨n2, n1 - n2, by omega, by
      have hEq : n2 + (n1 - n2) = n1 := by omega
      rw [hEq]; exact heq⟩

/-- Strengthened version: once two iterates coincide, they coincide from
that point on forever (not just at that one index). This is exactly what's
needed to conclude "`a(n) mod m` is eventually periodic for all `n ≥ N`",
not just for one particular `n`. -/
theorem eventually_periodic_of_finite' {α : Type*} [Finite α] (f : α → α)
    (x : α) : ∃ N p : ℕ, 0 < p ∧ ∀ n ≥ N, f^[n + p] x = f^[n] x := by
  obtain ⟨N, p, hp, hNp⟩ := eventually_periodic_of_finite f x
  refine ⟨N, p, hp, fun n hn => ?_⟩
  have h1 : f^[n] x = f^[n - N] (f^[N] x) := by
    rw [← Function.iterate_add_apply]; congr 1; omega
  have h2 : f^[n + p] x = f^[n - N] (f^[N + p] x) := by
    rw [← Function.iterate_add_apply]; congr 1; omega
  rw [h1, h2, hNp]

/- ================================================================
   6. Assembly: package Lemmas 1–3 as "the state transition is a
      well-defined self-map of a finite type", then invoke Lemma 4.
   ================================================================ -/

section Assembly
variable (m : ℕ)

/-- The state at position `n`: `n mod m` together with `b(n-i, g) mod m` for
the `m` preceding positions and every `g` in the (corrected) alphabet. This
is precisely the tuple described at the end of the write-up. -/
def State (m : ℕ) := ZMod m × (Fin m → {g // g ∈ Alphabet m} → ZMod m)

noncomputable instance : Fintype (State m) := by
  unfold State
  infer_instance

noncomputable def stateOf (n : ℕ) : State m :=
  ((n : ZMod m), fun i g => ((b (n - (i : ℕ)) g.1 : ℕ) : ZMod m))

/-- **Lemmas 1 + 2 + 3 combined.** The pruned recurrence `(⋆_m)` expresses
`b (n+1) g mod m`, for every `g` in the alphabet, purely in terms of
`stateOf m n` (it needs `n mod m` for the coefficients, by Lemma 1; it only
needs `j ≤ m`, by Lemma 2; and the accumulated `g`'s it needs stay inside the
alphabet, by Lemma 3). Hence `stateOf m (n+1)` is a function of `stateOf m
n` alone, for `n ≥ m`. This is the crux fact carrying all the *specific*
mathematical content of the write-up; everything after it is generic. -/
theorem transition_wellDefined (hm : 0 < m) :
    ∃ F : State m → State m, ∀ n, n ≥ m → stateOf m (n + 1) = F (stateOf m n) := by
  sorry

/-- **Main theorem**, assembled from `transition_wellDefined` (the
problem-specific part) and `eventually_periodic_of_finite'` (the generic
pigeonhole part). The two `sorry`s below are routine bookkeeping — unwinding
`stateOf` along the orbit of `F`, and reading the first coordinate back out
as a statement about `a` — rather than new mathematical content. -/
theorem A060014_eventually_periodic (hm : 0 < m) :
    ∃ N p : ℕ, 0 < p ∧ ∀ n ≥ N, (a (n + p) : ZMod m) = (a n : ZMod m) := by
  obtain ⟨F, hF⟩ := transition_wellDefined m hm
  have horbit : ∀ k, stateOf m (m + k) = F^[k] (stateOf m m) := by
    intro k
    induction k with
    | zero => simp
    | succ k ih =>
        have step := hF (m + k) (by omega)
        rw [show m + (k + 1) = (m + k) + 1 from by ring, step, ih]
  obtain ⟨N', p, hp, hper⟩ := eventually_periodic_of_finite' F (stateOf m m)
  refine ⟨m + N', p, hp, fun n hn => ?_⟩
  -- From here: rewrite `stateOf m (n+p)` and `stateOf m n` via `horbit`,
  -- apply `hper`, then extract the first coordinate (`n mod m` slot encodes
  -- `a n mod m` via `g = 1`, since `Alphabet m` always contains `1`).
  sorry

end Assembly